# transformer.ipynb

TCFN: temporal encoder + fusion module, first version (negative result).

Paths are relative: `data/` holds dialogues and embeddings (not included), `models/` holds fine-tuned weights (not included). See the README.


## Setup: models

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

device = "cuda" if torch.cuda.is_available() else "cpu"

llm_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(llm_name)

local_embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)

victim_embedder = SentenceTransformer("intfloat/e5-base-v2", device=device)

llm.to(device)
llm.eval()

In [ ]:
import copy
import torch.nn as nn
import torch.nn.functional as F

d_emb = 768
d_victim = 768
d_hist = 512
top_k = 100
beam_size = 5
prune_threshold = 0.2
learning_rate = 1e-2
seq_len_hist = 3
gpu_total_mem = torch.cuda.get_device_properties(0).total_memory
threshold = 0.8 * gpu_total_mem

## Modules: projector B, history encoder H, fusion

In [ ]:
class BProjector(nn.Module):
    def __init__(self, d, d_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, d)
        )
        self.norm = nn.LayerNorm(d)

    def forward(self, x):
        x = F.normalize(x, dim=-1)
        out = self.net(x)
        return self.norm(x + out)

class HModule(nn.Module):
    def __init__(self, d_in, d_hist):
        super().__init__()
        self.rnn = nn.GRU(d_in, d_hist, batch_first=True)

    def forward(self, seq):
        _, h = self.rnn(seq)
        return h.squeeze(0)

class FusionModule(nn.Module):
    def __init__(self, d_victim, d_hist):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_victim + d_hist + 1, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
        self.victim_proj = nn.Linear(768, 1)

    def forward(self, topk_logits, b_out, h, victim_emb):
        """
        topk_logits: [B, K]
        b_out:       [B, d_victim]
        h:           [B, d_hist]
        """

        B, K = topk_logits.shape
        b_expand = b_out
        h_expand = h
        logit_expand = topk_logits.unsqueeze(-1)

        fused_input = torch.cat([logit_expand, b_expand, h_expand], dim=-1)

        score = self.mlp(fused_input).squeeze(-1)
        victim_score = self.victim_proj(victim_emb).squeeze(-1)

        return score, victim_score

In [ ]:
B = BProjector(d_emb, d_victim).to(device)
optimizer_B = torch.optim.Adam(B.parameters(), lr=1e-2)

In [ ]:
H = HModule(d_emb, d_hist).to(device)

Fusion = FusionModule(d_victim, d_hist).to(device)
optimizer_HF = torch.optim.Adam(list(H.parameters()) + list(Fusion.parameters()), lr=1e-3)

In [ ]:
from torch.nn.utils.rnn import pad_sequence
import string

allowed_chars = set(string.ascii_letters + string.digits + " .,!?'\"-")

## Helpers

In [ ]:
def encode_local(texts):
    return local_embedder.encode(texts, convert_to_tensor=True).to(device)

def encode_victim(texts):
    return victim_embedder.encode(texts, convert_to_tensor=True).to(device)

def diversity_filter(token_embs, top_ids, KS=50, T_hw=0.9):
    top_k = token_embs.size(0)
    mask = torch.ones(top_k, dtype=torch.bool, device=token_embs.device)

    selected_indices = []
    backup_indices = []

    for i in range(top_k):
        token_id = top_ids[i].item()
        token_str = tokenizer.decode(token_id, skip_special_tokens=True)

        if not mask[i]:
            continue

        if all(c in allowed_chars for c in token_str):

            selected_indices.append(i)

            sims = F.cosine_similarity(token_embs[i].unsqueeze(0), token_embs, dim=1)
            sims[i] = 0
            mask = mask & (sims <= T_hw)

        else:

            backup_indices.append(i)

        if len(selected_indices) >= KS:
            break

    if len(selected_indices) < KS:
        needed = KS - len(selected_indices)

        selected_indices.extend(backup_indices[:needed])

    return selected_indices

## Training steps

In [ ]:
def train_step(target_embs, max_len):
    """
    target_embs: tensor [B, d_victim], the leaked embeddings, used by the score function
    """
    batch_size = len(target_embs)
    beams = [[" "] for _ in range(batch_size)]
    recovered_memory = [[] for _ in range(batch_size)]
    top_k = 1000
    KS = 100

    for t in range(max_len):
        batch_prefix_idx = []
        batch_prefix = []

        for b_idx, beam_list in enumerate(beams):
            for prefix in beam_list:
                batch_prefix_idx.append(b_idx)
                batch_prefix.append(prefix)

        if not batch_prefix:
            break

        batch_input_ids = tokenizer(
            batch_prefix, return_tensors="pt", padding=True
        ).input_ids.to(device)

        with torch.no_grad():
            batch_logits = llm(batch_input_ids).logits[:, -1, :]
        batch_log_probs = F.log_softmax(batch_logits, dim=-1)

        all_new_texts = []
        all_candidate_meta = []
        all_prob = []

        for i, b_idx in enumerate(batch_prefix_idx):
            prefix = batch_prefix[i]
            hist_seq = recovered_memory[b_idx][-seq_len_hist:]

            top_vals, top_ids = batch_log_probs[i].topk(top_k, dim=-1)

            with torch.no_grad():
                token_embs = llm.get_input_embeddings()(top_ids)

            diverse_indices = diversity_filter(token_embs, top_ids, KS=KS, T_hw=0.9)
            diverse_indices_tensor = torch.tensor(diverse_indices, device=device)
            prob = top_vals[diverse_indices_tensor]

            for idx in diverse_indices:
                token_id = top_ids[idx].item()
                log_prob = top_vals[idx].item()
                token_str = tokenizer.decode(token_id, skip_special_tokens=True)
                new_text = prefix + (" " if not token_str.startswith(" ") else "") + token_str
                all_new_texts.append(new_text)
                all_candidate_meta.append((b_idx, new_text, hist_seq))
            all_prob.append(prob)
            del prob

        if not all_new_texts:
            break

        all_embs = encode_local(all_new_texts).clone()
        all_pred_v = B(all_embs.clone())

        all_candidates = []
        all_new_text = []
        for k, (b_idx, new_text, hist_seq) in enumerate(all_candidate_meta):
            pred_v = all_pred_v[k].reshape(1, -1)
            target_v = target_embs[b_idx]
            all_new_text.append(new_text)
            all_candidates.append((b_idx, new_text, hist_seq, pred_v, target_v))

        if len(all_candidates) == 0:
            break
        print(f"Candidates length: {len(all_candidates)}")

        victim_true = encode_victim(all_new_text).clone()
        optimizer_B.zero_grad()
        pred = F.normalize(all_pred_v, dim=-1)
        target = F.normalize(victim_true, dim=-1)
        loss_B = 1 - F.cosine_similarity(pred, target, dim=-1).mean()
        loss_B.backward(retain_graph=True)

        print(f"Step {t}, loss_B: {loss_B.item():.4f}")

        new_all_candidates = []
        conf_t = min(1.0, (t+1)/max_len)

        texts = [x[1] for x in all_candidates]

        input_ids = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).input_ids.to(device)

        hist_tensors = []

        for _, _, hist_seq, _, _ in all_candidates:
            if hist_seq:
                hist_tensors.append(encode_local(hist_seq))
            else:
                hist_tensors.append(torch.zeros(1, 768))

        hist_tensor = torch.stack(hist_tensors).to(device)
        logits = torch.stack(all_prob).to(device)
        b, k = logits.shape

        h_proj = B(hist_tensor)
        h_out = H(h_proj)
        b_out = all_pred_v.view(b, k, -1).detach().clone()
        victim_true_flatten = victim_true.view(b, k, -1).clone()
        h_out = h_out.view(b, k, -1)
        fused_logits, victim_score = Fusion(logits, b_out, h_out, victim_true_flatten)

        fused_score = fused_logits.clone()
        optimizer_HF.zero_grad()
        F_mse_loss = F.mse_loss(fused_score, victim_score)
        fused_score = F.normalize(fused_score, dim=-1)
        victim_score = F.normalize(victim_score, dim=-1)
        F_cos_loss = 1 - F.cosine_similarity(fused_score, victim_score, dim=-1).mean()
        loss_Fusion = F_cos_loss + 0.2 * F_mse_loss

        loss_Fusion.backward()
        optimizer_B.step()
        optimizer_HF.step()

        print(f"Step {t}, loss_Fusion: {loss_Fusion.item():.4f}")

        target_v = torch.stack([torch.tensor(x[4]) for x in all_candidates]).to(device)
        cos_sim = F.cosine_similarity(victim_true, target_v, dim=-1)

        combined_score = fused_logits.view(-1).detach() + conf_t * cos_sim

        new_all_candidates = [
            (b_idx, text, combined_score[i].item())
            for i, (b_idx, text, _, _, _) in enumerate(all_candidates)
        ]

        del all_candidates, batch_input_ids, batch_prefix_idx, pred, target, victim_true_flatten
        mem_used = torch.cuda.memory_allocated()
        if mem_used > threshold:
            print(f"Step {t}: memory {mem_used/1e9:.2f} GB exceeds threshold, clearing cache")
            torch.cuda.empty_cache()

        beam_dict = {i: [] for i in range(batch_size)}
        for b_idx, new_text, combined_score in new_all_candidates:
            beam_dict[b_idx].append((new_text, combined_score))

        beams = []
        for b_idx in range(batch_size):
            top_candidates = sorted(beam_dict[b_idx], key=lambda x: x[1], reverse=True)[:beam_size]
            beams.append([(x[0]) for x in top_candidates])

            if top_candidates:
                recovered_memory[b_idx].append(top_candidates[0][0])
        print(f"Step {t}: {beams}\n")

    final_texts = [beam_list[0][0] if beam_list else "" for beam_list in beams]
    return final_texts

In [ ]:
def train_B_step(all_new_texts):
    max_len = len(all_new_texts)

    for t in range(max_len):
        all_embs = encode_local(all_new_texts).clone()
        all_pred_v = B(all_embs.clone())

        victim_true = encode_victim(all_new_texts).clone()
        optimizer_B.zero_grad()
        pred = F.normalize(all_pred_v, dim=-1)
        y = F.normalize(victim_true, dim=-1)
        loss_B = 1 - F.cosine_similarity(pred, y, dim=-1).mean()

        loss_B.backward()
        optimizer_B.step()
        print(f"Step {t}, loss_B: {loss_B.item():.4f}")

In [ ]:
def text_print(texts):
  for text in texts:
    print(text)

def train_B_dataset(dataset, num_epoch):
  result = []
  epoch = 0
  for i in num_epoch:
    item = dataset[i]
    batch_text = []
    for d in item["dialogue"]:
      text = d["text"]
      batch_text.append(text)
    train_B_step(batch_text)
    print(f"Epoch {epoch+1}/{len(num_epoch)} done.")
    epoch += 1

def train_dataset(dataset, num_epoch):
  result = []
  epoch = 0
  for i in num_epoch:
    item = dataset[i]
    batch_embed = []
    batch_text = []
    max_l = 0
    for d in item["dialogue"]:
      text = d["text"]
      l = len(text.split())
      if l > max_l:
        max_l = l
      batch_embed.append(d["embedding"])
      batch_text.append(text)
    gen_text = train_step(batch_embed, max_l)
    print(f"Epoch {epoch+1}/{num_epoch} done.")
    print("Reference Texts:")
    text_print(batch_text)
    print("Generated Texts:")
    text_print(gen_text)
    print("-"*200)
    epoch += 1
    result.append(gen_text)
    return result
    break

## Training runs

In [ ]:
import json
with open("data/ill_embed.json", "r") as f:
  ill_data = json.load(f)
# train the projector B on the first 100 medical conversations
extract = [i for i in range(100)]
train_B_dataset(ill_data, extract)

In [ ]:
torch.save(B, "models/B.pth")

In [ ]:
import json
with open("data/ill_embed.json", "r") as f:
  ill_data = json.load(f)
# subset of the medical conversations whose turns all have at most 17 words
extract = [1, 113, 151, 273, 404, 405, 454, 611, 614, 616, 619, 628, 633, 666, 723, 806, 814, 874, 875, 877]
result = train_dataset(ill_data, extract)
for r in result:
  print(r)

In [ ]:
torch.save(Fusion.state_dict(), "models/Fusion.pth")